# **JPM Q1 Data**

---

Notebook-only entrypoint for data refresh and smoke-test validation.


## 0. Global Configurations

---


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

QUESTION_ROOT = Path('..').resolve()
SCRIPTS_DIR = QUESTION_ROOT / 'scripts'

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import clean_financials as clean_fin
import fetch_sec_xbrl as sec_fetch
import fetch_yf_statements as yf_fetch

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

DATA_ROOT = QUESTION_ROOT / 'data'
OUTPUT_ROOT = QUESTION_ROOT / 'output'
CONFIG_ROOT = QUESTION_ROOT / 'config'

## 1. Data Update Controls

---

Use one sample ticker for a fast smoke test. This notebook is for refresh orchestration, not model analysis.


In [ ]:
SMOKE_TICKER = 'AAPL'
REFETCH_SAMPLE = True
RUN_YFINANCE = True
RUN_SEC = True
RUN_CLEAN = True
SEC_FREQ = 'both'
SEC_YEARS = 6
SEC_QUARTERS = 8
SEC_USER_AGENT = os.environ.get('SEC_USER_AGENT') or None

sample_ticker = yf_fetch.normalize_ticker(SMOKE_TICKER)
sample_slug = yf_fetch.slugify(sample_ticker)


pd.DataFrame([
    {
        'smoke_ticker': sample_ticker,
        'sample_slug': sample_slug,
        'refetch_sample': REFETCH_SAMPLE,
        'run_yfinance': RUN_YFINANCE,
        'run_sec': RUN_SEC,
        'run_clean': RUN_CLEAN,
        'sec_freq': SEC_FREQ,
        'sec_years': SEC_YEARS,
        'sec_quarters': SEC_QUARTERS,
    }
])

,smoke_ticker,sample_slug,refetch_sample,run_yfinance,run_sec,run_clean,sec_freq,sec_years,sec_quarters
0,AAPL,aapl,True,True,True,True,both,6,8


## 2. Yahoo Finance Refresh

---

Minimal fetch against the existing Yahoo Finance script. This touches one ticker only.


In [ ]:
yf_result = {}

if RUN_YFINANCE:
    yf_result = yf_fetch.refresh_ticker_data(
        SMOKE_TICKER,
        data_dir = DATA_ROOT / 'yfinance',
        force = REFETCH_SAMPLE,
    )

yf_summary = pd.DataFrame([yf_result]) if yf_result else pd.DataFrame()


yf_summary

,ticker,slug,status,yearly_rows,quarterly_rows,paths
0,AAPL,aapl,ok,5,6,[D:\OneDrive\Documents\Desktop\JPM\Question_1\...


## 3. SEC XBRL Refresh

---

Minimal fetch against the existing SEC XBRL script. AAPL uses the configured CIK, so no full ticker-map refresh is needed.


In [ ]:
sec_result = {}

if RUN_SEC:
    sec_result = sec_fetch.refresh_ticker_facts(
        SMOKE_TICKER,
        freq = SEC_FREQ,
        years = SEC_YEARS,
        quarters = SEC_QUARTERS,
        user_agent = SEC_USER_AGENT,
        data_dir_yearly = DATA_ROOT / 'sec' / 'yearly',
        data_dir_quarterly = DATA_ROOT / 'sec' / 'quarterly',
        cik_map = None,
    )

sec_summary = pd.DataFrame([sec_result]) if sec_result else pd.DataFrame()


sec_summary

,ticker,slug,cik,status,yearly_rows,quarterly_rows,paths
0,AAPL,aapl,0000320193,ok,6,8,[D:\OneDrive\Documents\Desktop\JPM\Question_1\...


## 4. Clean and Validate

---

Run the cleaning script only for the smoke-test ticker, then confirm the expected raw and clean files exist.


In [ ]:
clean_paths = []

if RUN_CLEAN:
    for freq in ('yearly', 'quarterly'):
        clean_paths.extend(str(path) for path in clean_fin.clean_yfinance_slug(sample_slug, freq))
        clean_paths.extend(str(path) for path in clean_fin.clean_sec_slug(sample_slug, freq))

expected_paths = [
    DATA_ROOT / 'yfinance' / 'yearly' / f'{sample_slug}_balance_sheet_yearly.csv',
    DATA_ROOT / 'yfinance' / 'yearly' / f'{sample_slug}_income_statement_yearly.csv',
    DATA_ROOT / 'yfinance' / 'quarterly' / f'{sample_slug}_balance_sheet_quarterly.csv',
    DATA_ROOT / 'yfinance' / 'quarterly' / f'{sample_slug}_income_statement_quarterly.csv',
    DATA_ROOT / 'sec' / 'yearly' / f'sec_{sample_slug}_yearly.csv',
    DATA_ROOT / 'sec' / 'quarterly' / f'sec_{sample_slug}_quarterly.csv',
    DATA_ROOT / 'clean' / 'yfinance' / 'yearly' / f'{sample_slug}_balance_sheet_yearly.csv',
    DATA_ROOT / 'clean' / 'yfinance' / 'yearly' / f'{sample_slug}_income_statement_yearly.csv',
    DATA_ROOT / 'clean' / 'yfinance' / 'quarterly' / f'{sample_slug}_balance_sheet_quarterly.csv',
    DATA_ROOT / 'clean' / 'yfinance' / 'quarterly' / f'{sample_slug}_income_statement_quarterly.csv',
    DATA_ROOT / 'clean' / 'sec' / 'yearly' / f'{sample_slug}_balance_sheet_yearly.csv',
    DATA_ROOT / 'clean' / 'sec' / 'yearly' / f'{sample_slug}_income_statement_yearly.csv',
    DATA_ROOT / 'clean' / 'sec' / 'quarterly' / f'{sample_slug}_balance_sheet_quarterly.csv',
    DATA_ROOT / 'clean' / 'sec' / 'quarterly' / f'{sample_slug}_income_statement_quarterly.csv',
]

validation_df = pd.DataFrame(
    {
        'path': [str(path) for path in expected_paths],
        'exists': [path.exists() for path in expected_paths],
    }
)


print(f'clean_file_count={len(clean_paths)}')
validation_df

clean_file_count=8


,path,exists
0,D:\OneDrive\Documents\Desktop\JPM\Question_1\d...,True
1,D:\OneDrive\Documents\Desktop\JPM\Question_1\d...,True
2,D:\OneDrive\Documents\Desktop\JPM\Question_1\d...,True
3,D:\OneDrive\Documents\Desktop\JPM\Question_1\d...,True
4,D:\OneDrive\Documents\Desktop\JPM\Question_1\d...,True
5,D:\OneDrive\Documents\Desktop\JPM\Question_1\d...,True
6,D:\OneDrive\Documents\Desktop\JPM\Question_1\d...,True
7,D:\OneDrive\Documents\Desktop\JPM\Question_1\d...,True
8,D:\OneDrive\Documents\Desktop\JPM\Question_1\d...,True
9,D:\OneDrive\Documents\Desktop\JPM\Question_1\d...,True
